In [ ]:
# Take in DOI of a paper and then create a knowledge graph based on citations and the relation to each of the other doi's based on content and context of citation

In [1]:
pip install pypdf2

  Obtaining dependency information for pypdf2 from https://files.pythonhosted.org/packages/8e/5e/c86a5643653825d3c913719e788e41386bee415c2b87b4f955432f2de6b2/pypdf2-3.0.1-py3-none-any.whl.metadata
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.6/232.6 kB 4.0 MB/s eta 0:00:00 0:00:01
Note: you may need to restart the kernel to use updated packages.


In [3]:
pip install spacy

  Obtaining dependency information for spacy from https://files.pythonhosted.org/packages/90/95/0823540c856b61193cb2d0c8feb863d5130e1571c79140317004ad786612/spacy-3.8.4-cp311-cp311-macosx_11_0_arm64.whl.metadata
  Obtaining dependency information for spacy-legacy<3.1.0,>=3.0.11 from https://files.pythonhosted.org/packages/c3/55/12e842c70ff8828e34e543a2c7176dac4da006ca6901c9e8b43efab8bc6b/spacy_legacy-3.0.12-py2.py3-none-any.whl.metadata
  Obtaining dependency information for spacy-loggers<2.0.0,>=1.0.0 from https://files.pythonhosted.org/packages/33/78/d1a1a026ef3af911159398c939b1509d5c36fe524c7b644f34a5146c4e16/spacy_loggers-1.0.5-py3-none-any.whl.metadata
  Obtaining dependency information for murmurhash<1.1.0,>=0.28.0 from https://files.pythonhosted.org/packages/2f/a4/a387486e79bcc04f3d3b123195fd4cca74a7ba439d6c45b35c5366c66586/murmurhash-1.0.12-cp311-cp311-macosx_11_0_arm64.whl.metadata
  Obtaining dependency information for cymem<2.1.0,>=2.0.2 from https://files.pythonhosted.org/p

In [2]:
import requests
import networkx as nx
from typing import Dict, List, Tuple
import spacy
from PyPDF2 import PdfReader
import json

class CitationKnowledgeGraph:
    def __init__(self):
        self.graph = nx.DiGraph()
        self.nlp = spacy.load("en_core_web_sm")
        
    def fetch_paper_metadata(self, doi: str) -> Dict:
        """Fetch paper metadata from Crossref API"""
        base_url = "https://api.crossref.org/works/"
        headers = {"Accept": "application/json"}
        response = requests.get(f"{base_url}{doi}", headers=headers)
        return response.json()["message"]
    
    def get_full_text_url(self, doi: str) -> str:
        """Get open access PDF URL using Unpaywall API"""
        email = "your-email@institution.edu"  # Required by Unpaywall
        base_url = f"https://api.unpaywall.org/v2/{doi}?email={email}"
        response = requests.get(base_url)
        data = response.json()
        
        if data.get("is_oa") and data.get("best_oa_location"):
            return data["best_oa_location"]["url_for_pdf"]
        return None
    
    def extract_citation_contexts(self, pdf_path: str) -> List[Dict]:
        """Extract citation contexts from PDF"""
        citation_contexts = []
        pdf = PdfReader(pdf_path)
        
        # Extract text from PDF
        full_text = ""
        for page in pdf.pages:
            full_text += page.extract_text()
            
        # Simple citation pattern matching (would need to be more sophisticated)
        # This is a basic example - you'd want more robust citation extraction
        doc = self.nlp(full_text)
        
        # Find sentences containing citations
        for sent in doc.sents:
            if "[" in sent.text and "]" in sent.text:  # Basic citation marker detection
                citation_contexts.append({
                    "text": sent.text,
                    "citation_markers": self._extract_citation_markers(sent.text)
                })
                
        return citation_contexts
    
    def analyze_citation_context(self, context: str) -> Dict:
        """Analyze the context of a citation to determine relationship type"""
        doc = self.nlp(context)
        
        relationships = {
            "uses_method_from": ["used", "applied", "implemented", "adopted"],
            "builds_upon": ["extends", "builds on", "improves", "enhances"],
            "supports": ["confirms", "validates", "supports", "agrees with"],
            "contrasts_with": ["differs from", "contradicts", "disputes", "challenges"]
        }
        
        # Basic rule-based classification
        for rel_type, indicators in relationships.items():
            if any(indicator in context.lower() for indicator in indicators):
                return rel_type
                
        return "cites"  # Default relationship
    
    def build_graph(self, starting_doi: str, depth: int = 1):
        """Build the citation knowledge graph starting from a given DOI"""
        papers_to_process = [(starting_doi, 0)]
        processed_dois = set()
        
        while papers_to_process:
            current_doi, current_depth = papers_to_process.pop(0)
            
            if current_doi in processed_dois or current_depth > depth:
                continue
                
            try:
                # Fetch metadata
                metadata = self.fetch_paper_metadata(current_doi)
                self.graph.add_node(current_doi, title=metadata.get("title", [""])[0])
                
                # Get full text if available
                pdf_url = self.get_full_text_url(current_doi)
                if pdf_url:
                    citation_contexts = self.extract_citation_contexts(pdf_url)
                    
                    # Process references
                    for reference in metadata.get("reference", []):
                        if reference.get("DOI"):
                            ref_doi = reference["DOI"]
                            
                            # Find context for this citation
                            context = self._find_citation_context(citation_contexts, reference)
                            relationship = self.analyze_citation_context(context) if context else "cites"
                            
                            # Add to graph
                            self.graph.add_edge(
                                current_doi,
                                ref_doi,
                                relationship=relationship,
                                context=context
                            )
                            
                            if current_depth < depth:
                                papers_to_process.append((ref_doi, current_depth + 1))
                
                processed_dois.add(current_doi)
                
            except Exception as e:
                print(f"Error processing DOI {current_doi}: {str(e)}")
    
    def export_graph(self, output_path: str):
        """Export the knowledge graph to a JSON file"""
        graph_data = nx.node_link_data(self.graph)
        with open(output_path, 'w') as f:
            json.dump(graph_data, f, indent=2)
    
    def _extract_citation_markers(self, text: str) -> List[str]:
        """Extract citation markers from text"""
        # Implement more sophisticated citation marker extraction
        # This is a basic example
        markers = []
        in_citation = False
        current_marker = ""
        
        for char in text:
            if char == "[":
                in_citation = True
                current_marker = "["
            elif char == "]" and in_citation:
                in_citation = False
                current_marker += "]"
                markers.append(current_marker)
            elif in_citation:
                current_marker += char
                
        return markers
    
    def _find_citation_context(self, contexts: List[Dict], reference: Dict) -> str:
        """Find the context for a specific citation"""
        # Implement more sophisticated context matching
        # This is a basic example
        ref_author = reference.get("author", "")
        ref_year = reference.get("year", "")
        
        for context in contexts:
            if ref_author in context["text"] and str(ref_year) in context["text"]:
                return context["text"]
        
        return None

# Usage example:
if __name__ == "__main__":
    graph = CitationKnowledgeGraph()
    starting_doi = "10.1234/example.doi"
    graph.build_graph(starting_doi, depth=2)
    graph.export_graph("citation_knowledge_graph.json")

ModuleNotFoundError: No module named 'spacy'

In [1]:
# scihub experiment